In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in data_loader:
          images = images.to(device)
          labels = labels.to(device)
          outputs = model(images)
          loss = criterion(outputs, labels)
          preds = outputs.argmax(dim=1)
          total_loss += loss.item() * images.size(0)
          correct += (preds == labels).sum().item()
          total += labels.size(0)
          all_labels.extend(labels.cpu().numpy())
          all_preds.extend(preds.cpu().numpy())
    return total_loss / total, correct / total, all_labels, all_preds

def get_f1_scores(labels, preds, class_names):
    report = classification_report(
        labels,
        preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    macro_f1 = report["macro avg"]["f1-score"]
    weighted_f1 = report["weighted avg"]["f1-score"]
    return macro_f1, weighted_f1

In [ ]:
def train_model(model, train_loader, val_loader, test_loader, class_names, config, checkpoint_path):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=config["learning_rate"])
    if USE_WANDB:
        wandb.init(
            project=PROJECT_NAME,
            name=config["model_name"],
            config=config
        )
    best_val_acc = -1
    best_epoch = 0
    for epoch in range(config["num_epochs"]):
        train_loss, train_acc = train_one_epoch(model,train_loader,criterion,optimizer,device)
        val_loss, val_acc, _, _ = evaluate(model,val_loader,criterion,device)
        if USE_WANDB:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc
            })
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_to_idx": train_dataset.class_to_idx,
                "config": config,
                "best_epoch": best_epoch,
                "best_val_accuracy": best_val_acc
            }, checkpoint_path)
        print(
            f"Epoch [{epoch + 1}/{config['num_epochs']}] "
            f"train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, "
            f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}",
            flush=True
        )
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_loss, test_acc, test_labels, test_preds = evaluate(model,test_loader,criterion,device)
    macro_f1, weighted_f1 = get_f1_scores(test_labels,test_preds,class_names)
    if USE_WANDB:
        wandb.log({
            "best_epoch": best_epoch,
            "best_val_accuracy": best_val_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "test_macro_f1": macro_f1,
            "test_weighted_f1": weighted_f1
        })
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_accuracy"] = best_val_acc
        wandb.summary["test_loss"] = test_loss
        wandb.summary["test_accuracy"] = test_acc
        wandb.summary["test_macro_f1"] = macro_f1
        wandb.summary["test_weighted_f1"] = weighted_f1

    return {
        "model": config["model_title"],
        "best_val_accuracy": best_val_acc,
        "test_accuracy": test_acc,
        "test_macro_f1": macro_f1,
        "test_weighted_f1": weighted_f1,
        "test_labels": test_labels,
        "test_preds": test_preds,
        "checkpoint_path": checkpoint_path
    }

In [ ]:
def log_model_artifact(checkpoint_path, artifact_name):
    if not USE_WANDB:
        return
    artifact = wandb.Artifact(name=artifact_name,type="model",description=f"Best checkpoint for {artifact_name}")
    artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(artifact)
    wandb.finish()
    print("Model artifact logged:", artifact_name)